# Langchain 알아보기
- Part 1. Langchain - Prompt template
- Part 2. Langchain - Chain
- Part 3. Langchain - VectorDB(Pinecone) 활용한 RAG
- Part 4. Langchain - Agent

### Part 1. Langchain - Prompt Template
- Objectives: Langchain 라이브러리를 활용하여 GPT 모델에 대한 프롬프트 엔지니어링 진행


In [7]:
# !pip install -U langchain langchain-openai python-dotenv

In [8]:
# 아주 어썸한 소설을 써줘
import os
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    raise ValueError('OPENAI_API_KEY not found in .env')

# 프롬프트 엔지니어링이란?
### LLM 프롬프트 수정을 통해 일반 바닐라한 기능성에 대해 추가적인 기능을 부여
- 1) 인스트럭션
> LLM에게 무엇을 하기를 원하는지 요건을 적는 필드
- 2) 컨텍스트
> 답변 생성 시 참고하여 답을 하기를 원하는 정보를 적는 필드
- 3) 사용자 쿼리
> 사용자의 질문을 넣는 필드(템플릿 전체가 질문일 순 없으니)
- 4) 답변 필드
> 질문에 따른 답변을 하는 필드

In [9]:

prompt = """Please answer the question based on the context below. If you can't find the information to answer the question
from the context provided, please say "I don't know".

context: Bitcoin is the first decentralized cryptocurrency.
Nodes in the peer-to-peer bitcoin network verify transactions through cryptography and record them in a public distributed ledger,
called a blockchain, without central oversight.
Consensus between nodes is achieved using a computationally intensive process based on proof of work,
called mining, that requires increasing quantities of electricity and guarantees the security of the bitcoin blockchain.
Based on a free market ideology, bitcoin was invented in 2008 by Satoshi Nakamoto, an unknown person.
Use of bitcoin as a currency began in 2009, ith the release of its open-source implementation.
Bitcoin is currently used more as a store of value and less as a medium of exchange or unit of account.
It is mostly seen as an investment and has been described by many scholars as an economic bubble.
As bitcoin is pseudonymous, its use by criminals has attracted the attention of regulators,
leading to its ban by several countries as of 2021.

Question: how is bitcoin used?

Answer:"""

In [ ]:
#랭체인에서 일반 채팅 모델(gpt completion)활용한 간단한 질의 구성 래퍼
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(
    api_key=openai_api_key,
    model="gpt-4.1-mini",
    temperature=0
)

In [11]:
# invoke통한 사전정의 prompt 기반 llm 답변 생성
print(chat.invoke(prompt).content)

Bitcoin is currently used more as a store of value and less as a medium of exchange or unit of account. It is mostly seen as an investment.


In [12]:
#이제 프롬프트 템플릿을 사용해서 사용자 쿼리에 따라 프롬프트가 자동으로 업데이트 되는 구조를 구현
from langchain_core.prompts import PromptTemplate

template = """Please answer the question based on the context below. If you can't find the information to answer the question
from the context provided, please say "I don't know".

Context: Bitcoin is the first decentralized cryptocurrency.
Nodes in the peer-to-peer bitcoin network verify transactions through cryptography and record them in a public distributed ledger,
called a blockchain, without central oversight.
Consensus between nodes is achieved using a computationally intensive process based on proof of work,
called mining, that requires increasing quantities of electricity and guarantees the security of the bitcoin blockchain.
Based on a free market ideology, bitcoin was invented in 2008 by Satoshi Nakamoto, an unknown person.
Use of bitcoin as a currency began in 2009, ith the release of its open-source implementation.
Bitcoin is currently used more as a store of value and less as a medium of exchange or unit of account.
It is mostly seen as an investment and has been described by many scholars as an economic bubble.
As bitcoin is pseudonymous, its use by criminals has attracted the attention of regulators,
leading to its ban by several countries as of 2021.

Question: {query}

Answer: """

prompt_template = PromptTemplate(
    input_variables=["query"],
    template=template
)

In [13]:
print(
    prompt_template.format(
        query="what is bitcoin?"
    )
)

Please answer the question based on the context below. If you can't find the information to answer the question
from the context provided, please say "I don't know".

Context: Bitcoin is the first decentralized cryptocurrency.
Nodes in the peer-to-peer bitcoin network verify transactions through cryptography and record them in a public distributed ledger,
called a blockchain, without central oversight.
Consensus between nodes is achieved using a computationally intensive process based on proof of work,
called mining, that requires increasing quantities of electricity and guarantees the security of the bitcoin blockchain.
Based on a free market ideology, bitcoin was invented in 2008 by Satoshi Nakamoto, an unknown person.
Use of bitcoin as a currency began in 2009, ith the release of its open-source implementation.
Bitcoin is currently used more as a store of value and less as a medium of exchange or unit of account.
It is mostly seen as an investment and has been described by many scho

In [14]:
print(
    chat.invoke(prompt_template.format(
        query="What is bitcoin?"
    )).content
)

Bitcoin is the first decentralized cryptocurrency that operates on a peer-to-peer network where nodes verify transactions through cryptography and record them in a public distributed ledger called a blockchain, without central oversight. It was invented in 2008 by an unknown person named Satoshi Nakamoto and began use as a currency in 2009. Bitcoin is based on a free market ideology and uses a computationally intensive proof-of-work process called mining to achieve consensus and secure the blockchain. It is currently used more as a store of value and investment rather than as a medium of exchange or unit of account.


In [15]:
# 한글 작동 확인
print(
    chat.invoke(prompt_template.format(
        query="비트코인이 뭐야?"
    )).content
)

비트코인은 최초의 탈중앙화된 암호화폐입니다. 중앙 관리자가 없이 피어 투 피어 네트워크의 노드들이 암호학을 통해 거래를 검증하고, 이를 블록체인이라는 공개 분산 원장에 기록합니다. 비트코인은 2008년에 사토시 나카모토라는 알려지지 않은 인물에 의해 발명되었으며, 2009년에 오픈 소스 구현이 공개되면서 사용되기 시작했습니다. 현재 비트코인은 주로 가치 저장 수단으로 사용되며, 투자 대상으로 여겨지고 있습니다.


In [16]:
# 프롬프트 수정 - 한국말로만 내뱉게끔
template = """Please answer the question based on the context below. If you can't find the information to answer the question
from the context provided, please say "I don't know". You must always answer in Korean Language.

Context: Bitcoin is the first decentralized cryptocurrency.
Nodes in the peer-to-peer bitcoin network verify transactions through cryptography and record them in a public distributed ledger,
called a blockchain, without central oversight.
Consensus between nodes is achieved using a computationally intensive process based on proof of work,
called mining, that requires increasing quantities of electricity and guarantees the security of the bitcoin blockchain.
Based on a free market ideology, bitcoin was invented in 2008 by Satoshi Nakamoto, an unknown person.
Use of bitcoin as a currency began in 2009, ith the release of its open-source implementation.
Bitcoin is currently used more as a store of value and less as a medium of exchange or unit of account.
It is mostly seen as an investment and has been described by many scholars as an economic bubble.
As bitcoin is pseudonymous, its use by criminals has attracted the attention of regulators,
leading to its ban by several countries as of 2021.

Question: {query}

Answer: """

prompt_template = PromptTemplate(
    input_variables=["query"],
    template=template
)
print(
    chat.invoke(prompt_template.format(
        query="What is bitcoin?"
    )).content
)

비트코인은 최초의 탈중앙화된 암호화폐입니다.


In [17]:
# 프롬프트 수정 - 동화책 읽는 말투로
template = """Please answer the question based on the context below. If you can't find the information to answer the question
from the context provided, please say "I don't know". You must always answer in Korean language, as in fairy tale reading style.

Context: Bitcoin is the first decentralized cryptocurrency.
Nodes in the peer-to-peer bitcoin network verify transactions through cryptography and record them in a public distributed ledger,
called a blockchain, without central oversight.
Consensus between nodes is achieved using a computationally intensive process based on proof of work,
called mining, that requires increasing quantities of electricity and guarantees the security of the bitcoin blockchain.
Based on a free market ideology, bitcoin was invented in 2008 by Satoshi Nakamoto, an unknown person.
Use of bitcoin as a currency began in 2009, ith the release of its open-source implementation.
Bitcoin is currently used more as a store of value and less as a medium of exchange or unit of account.
It is mostly seen as an investment and has been described by many scholars as an economic bubble.
As bitcoin is pseudonymous, its use by criminals has attracted the attention of regulators,
leading to its ban by several countries as of 2021.

Question: {query}

Answer: """

prompt_template = PromptTemplate(
    input_variables=["query"],
    template=template
)
print(
    chat.invoke(prompt_template.format(
        query="What is bitcoin?"
    )).content
)

옛날 옛적에, 비트코인이라는 마법의 동전이 있었어요. 이 동전은 중앙의 왕이나 관리자가 없이, 모두가 함께 지키고 확인하는 신비한 네트워크 속에서 살아갔지요. 이 네트워크의 작은 요정들, 즉 노드들이 암호라는 마법을 사용해 거래를 확인하고, 모두가 볼 수 있는 커다란 책, 블록체인에 기록했답니다. 이 마법의 힘은 '작업 증명'이라는 어려운 시험을 통과해야만 얻을 수 있었고, 그 과정은 많은 전기의 힘을 필요로 했지요. 2008년에 사토시 나카모토라는 신비로운 인물이 이 비트코인을 세상에 내놓았고, 2009년부터 사람들은 이 동전을 사용하기 시작했답니다. 비트코인은 주로 보물처럼 소중히 간직하는 가치의 저장고로 쓰였고, 어떤 이들은 그것을 거품이라 부르기도 했어요. 하지만 이 동전은 익명의 힘을 지니고 있어, 때로는 나쁜 이들의 관심을 끌기도 했답니다. 그래서 몇몇 나라에서는 이 마법의 동전을 금지하기도 했지요. 바로 이것이 비트코인이란 이야기였어요.


In [18]:
# Few-shot으로 톤조정하기
template = """Please answer the question based on the context below. If you can't find the information to answer the question
from the context provided, please say "I don't know". You must always answer in Korean language, as in fairy tale reading style.

Context: Bitcoin is the first decentralized cryptocurrency.
Nodes in the peer-to-peer bitcoin network verify transactions through cryptography and record them in a public distributed ledger,
called a blockchain, without central oversight.
Consensus between nodes is achieved using a computationally intensive process based on proof of work,
called mining, that requires increasing quantities of electricity and guarantees the security of the bitcoin blockchain.
Based on a free market ideology, bitcoin was invented in 2008 by Satoshi Nakamoto, an unknown person.
Use of bitcoin as a currency began in 2009, ith the release of its open-source implementation.
Bitcoin is currently used more as a store of value and less as a medium of exchange or unit of account.
It is mostly seen as an investment and has been described by many scholars as an economic bubble.
As bitcoin is pseudonymous, its use by criminals has attracted the attention of regulators,
leading to its ban by several countries as of 2021.

Question: When was bitcoin first used?
Answer: 비트코인은 2009년에 처음 사용되기 시작했답니다~

Question: Who invented Bitcoin?
Answer: 비트코인은 2008년에 사토시 나카모토라고 하는 신원미상의 사람이 발명한 코인이랍니다~

Question: {query}

Answer: """

prompt_template = PromptTemplate(
    input_variables=["query"],
    template=template
)

print(
    chat.invoke(prompt_template.format(
        query="What is bitcoin?"
    )).content
)

비트코인은 중앙의 감독 없이, 피어 투 피어 네트워크 속 노드들이 암호학으로 거래를 검증하고, 모두가 함께 나누는 공공 분산 원장인 블록체인에 기록하는 최초의 탈중앙화된 암호화폐란다~


In [19]:
#랭체인 퓨샷프롬프트탬플릿으로 더 간편하게 만들기

from langchain_core.prompts import FewShotPromptTemplate
examples = [
    {
        "query" : "When was bitcoin first used?",
        "answer" : "비트코인은 2009년에 처음 사용되기 시작했답니다~"
    },{
        "query": "Who invented Bitcoin?",
        "answer": "비트코인은 2008년에 사토시 나카모토라고 하는 신원미상의 사람이 발명한 코인이랍니다~"
    }
]

example_template = """
question: {query}
answer: {answer}
"""

example_prompt = PromptTemplate(
    input_variables=["query", "answer"],
    template=example_template
)

prefix = """Please answer the question based on the context below. If you can't find the information to answer the question
from the context provided, please say "I don't know". You must always answer in Korean language, as in fairy tale reading style.

Context: Bitcoin is the first decentralized cryptocurrency.
Nodes in the peer-to-peer bitcoin network verify transactions through cryptography and record them in a public distributed ledger,
called a blockchain, without central oversight.
Consensus between nodes is achieved using a computationally intensive process based on proof of work,
called mining, that requires increasing quantities of electricity and guarantees the security of the bitcoin blockchain.
Based on a free market ideology, bitcoin was invented in 2008 by Satoshi Nakamoto, an unknown person.
Use of bitcoin as a currency began in 2009, ith the release of its open-source implementation.
Bitcoin is currently used more as a store of value and less as a medium of exchange or unit of account.
It is mostly seen as an investment and has been described by many scholars as an economic bubble.
As bitcoin is pseudonymous, its use by criminals has attracted the attention of regulators,
leading to its ban by several countries as of 2021.
"""

suffix = """
question: {query}
answer: """

few_shot_prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["query"],
    example_separator="\n\n"
)

In [20]:
query = "What is bitcoin?"
print(
    chat.invoke(few_shot_prompt_template.format(
        query="What is bitcoin?"
    )).content
)


비트코인은 중앙의 감독 없이 피어 투 피어 네트워크 속에서 암호학으로 거래를 검증하고, 모두가 볼 수 있는 분산 원장인 블록체인에 기록하는 최초의 탈중앙화된 암호화폐란다~
